In [ ]:
# !pip install......

# !pip install pandas numpy matplotlib
# !pip install japanize-matplotlib 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # 日本語フォント対応
import seaborn as sns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from scipy.signal import hilbert

import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from scipy.stats import zscore


def prep_data_for_stl(raw_df: pd.DataFrame, meta_col_count: int = 3) -> pd.DataFrame:
    """
    Cleans raw DataFrame and transposes it into a Time x Items format.
    """
    # 1. Extract and Transpose
    df_ts = raw_df.set_index('Item').iloc[:, meta_col_count:]
    df_t = df_ts.T

    # 2. Index Formatting
    df_t.index = pd.to_datetime(df_t.index)
    df_t = df_t.sort_index()

    # 3. Type Casting & Handle Missing Data (STL requires no NaNs)
    df_t = df_t.astype(np.float64)
    # df_t = df_t.ffill().bfill() # Forward/backward fill any missing percentages
    
    return df_t

def get_residuals(series: pd.Series, period: int = 12) -> pd.Series:
    """
    Extracts the residual component from a time series using STL.
    """
    # robust=True handles outliers (like COVID-19 anomalies) better
    res = STL(series, period=period, robust=True).fit()
    return res.resid

def get_detrended(series: pd.Series, period: int = 12) -> pd.Series:
    """
    トレンド(T)のみを除去し、S + R 成分を返す
    """
    res = STL(series, period=period, robust=True).fit()
    # 元のデータ(x)からトレンド(T)を引く = S + R
    return series - res.trend

# STEP 0: 読み込みと前処理
df = pd.read_csv('../Data_time_series/en_setai_over2_monthly.csv')
import os
import pandas as pd

root_dir = os.getcwd() 
root_dir = os.path.dirname(root_dir)

data_file = os.path.join(root_dir,'Data_time_series', 'en_share_over2_monthly.csv')
df_share = pd.read_csv(data_file)

# Filter for rows where Medium (中分類) and Minor (小分類) are '-'
# This isolates the top-level "Major" rows

major_rows = df[ (df['Category level 1'] != '-') & (df['Category level 2'] == '-') & (df['Category level 3'] == '-') ]

major_dict = dict(zip(major_rows['Category level 1'], major_rows['Item']))
print(major_dict['1']) # Output: 食料

hierarchy = {}

# Iterate through every row
for index, row in df.iterrows():
    major_id = row['Category level 1']
    med_id = row['Category level 2']
    min_id = row['Category level 3']
    name = row['Item']

    # Skip header/garbage rows if any
    if major_id == '-': continue 

    # 1. Initialize Major Level
    # We use the ID as the key, but store the Name inside
    if major_id not in hierarchy:
        # Look up the major name from our simple dict
        major_name = major_dict.get(major_id, "他")
        hierarchy[major_id] = {'name': major_name, 'med': {}}

    # 2. Add Medium Level (if this row represents a Medium category or deeper)
    if med_id != '-':
        if med_id not in hierarchy[major_id]['med']:
             # Use the name if this is the defining row, otherwise generic placeholder until found
            hierarchy[major_id]['med'][med_id] = {'name': name, 'small': {}}
        
        # Update name if this is exactly the Medium definition row
        if min_id == '-':
            hierarchy[major_id]['med'][med_id]['name'] = name

    # 3. Add Minor Level (if this row represents a Minor category)
    if min_id != '-':
        # Add the minor category to the medium's children
        hierarchy[major_id]['med'][med_id]['small'][min_id] = name
# Usage:
# hierarchy[1]['children'][1]['name']  -> Access the name of Medium Category 1 inside Major 1
# Initialize the flat dictionary
# Key = Item Name (e.g., 'パン'), Value = Formatted String (e.g., '食料ーパン')

name_to_path_map = {}

# Iterate through the hierarchy
for major_id, major_data in hierarchy.items():
    major_name = major_data['name']
    
    # 1. Add the Major category itself (Optional, if needed)
    # name_to_path_map[major_name] = major_name
    
    # Check if 'med' exists
    if 'med' in major_data:
        for med_id, med_data in major_data['med'].items():
            med_name = med_data['name']
            
            # 2. Add Medium Category: Input '穀類' -> Output '食料ー穀類'
            name_to_path_map[med_name] = f"{major_name}ー{med_name}"
            
            # Check if 'small' exists
            if 'small' in med_data:
                for small_id, small_name in med_data['small'].items():
                    # 3. Add Small Category: Input 'パン' -> Output '食料ーパン'
                    # Note: Using strict user format "Major-Small". 
                    # If you wanted full path "Major-Med-Small", use: f"{major_name}ー{med_name}ー{small_name}"
                    name_to_path_map[small_name] = f"{major_name}ー{med_name}ー{small_name}"

# Check the result
print(name_to_path_map['bread']) 
# Output: '食料ーパン'

def get_category_path(item_name):
    return name_to_path_map.get(item_name, item_name) # Returns "他" if not found

print(get_category_path('bread'))
# --- Step 1: Flatten your hierarchy dictionary ---
# This converts: {'1': {'name': 'food', 'med': {'1': {'name': 'grains', 'small': {'1': 'rice'...
# To: {'rice': 'food', 'bread': 'food', ...}

category_map = {}

for l1_id, l1_info in hierarchy.items():
    l1_name = l1_info['name']
    for l2_id, l2_info in l1_info['med'].items():
        # Option A: Map to Level 1 (Food vs Transport)
        # Option B: Map to Level 2 (Grains vs Meat)
        # Let's map to Level 1 for a high-level systemic view
        for l3_id, l3_name in l2_info['small'].items():
            category_map[l3_name] = l1_name

# Convert to a Series for easy grouping
category_series = pd.Series(category_map)

# Dataframe for 3 levels of category hierarchy

c1 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']=='-') & (df_share['Category level 3']=='-')]
c2 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']!='-') & (df_share['Category level 3']=='-')]
c3 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']!='-') & (df_share['Category level 3']!='-')]

# STEP 1: データ整形
df_clean = prep_data_for_stl(c3, meta_col_count=3)

# STEP 2: トレンドを除去し、Noisy Cycle (S + R) を抽出
df_noisy_cycle = df_clean.apply(get_detrended, period=12)

# STEP 3: 標準化 (z-score または Min-Max)
# 解析の特性上、平均0, 分散1の z-score が Phase Portrait には向いています
# df_target = df_noisy_cycle.apply(lambda x: (x - x.mean()) / x.std())
df_target = df_noisy_cycle.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=0)

# --- データの可視化 ---

for ca in df_target.columns:
    time_axis = df_target.index 
    x_data = df_target[ca].values
    
    # 物理的ポイント：バンドパスフィルタはあえて通さない
    # バンドパスを通すと R(t) の摂動成分が消えてしまい、ただの綺麗な円になってしまうため。
    x_centered = x_data - np.mean(x_data)

    # ヒルベルト変換を実行
    z_signal = hilbert(x_centered)
    amplitude = np.abs(z_signal)
    unwrapped_phase = np.unwrap(np.angle(z_signal))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(ca, fontsize=24)
    
    # --- 図A: 元の波形と振幅エンベロープ ---
    axes[0].plot(time_axis, x_centered, label='S(t) + R(t)', color='gray', alpha=0.5)
    axes[0].plot(time_axis, amplitude, label='Amplitude A(t)', color='red', linewidth=1.5)
    axes[0].set_title('(A) Noisy Seasonal Signal', fontsize=18)
    axes[0].legend()

    # --- 図B: Phase Portrait (あなたのスケッチの再現) ---
    # 理想的なリミットサイクル（Sのみ）の推定として平均半径を計算しても面白いです
    axes[1].plot(np.real(z_signal), np.imag(z_signal), 
                color='blue', linewidth=0.6, alpha=0.4, zorder=1)
    
    # 時間経過をグラデーションで表示
    scatter = axes[1].scatter(np.real(z_signal), np.imag(z_signal), 
                            c=np.arange(len(z_signal)), cmap='viridis', 
                            s=10, alpha=0.8, zorder=2)
    
    axes[1].set_title('(B) Noisy Limit Cycle', fontsize=18)
    axes[1].set_xlabel('Real ($S+R$)', fontsize=15)
    axes[1].set_ylabel('Imaginary', fontsize=15)
    axes[1].set_aspect('equal') # 円を正円に見せるために重要
    plt.colorbar(scatter, ax=axes[1], label='Time Steps')

    # --- 図C: 累積位相の進化 ---
    # 傾き(ω)が一定なら安定。ノイズが強いとガタガタする
    axes[2].plot(time_axis, unwrapped_phase, color='purple', linewidth=2)
    axes[2].set_title('(C) Phase Evolution', fontsize=18)
    
    plt.tight_layout()
    plt.show()